# Lab 047 — SAINT: make rows attend, then audit the context
[Lesson](../lessons/0047-saint.html) · [Primary paper](https://arxiv.org/abs/2106.01342) §3.2, Algorithm 1; §4, Eq.5.

**Skill:** implement intersample attention and measure how evaluation companions affect a prediction.
PROVIDED = read/run; TODO = your implementation; CHECK = immediate feedback; EXIT = paste your evidence.
CPU is sufficient for the learning lab (about a minute on the authoring machine; allow several minutes elsewhere).

**Mirror scope:** full supervised released-code path, plus the contrastive loss as a key-parts exercise.
The supervised model does not pretrain: the paper’s semi-supervised results remain untested.
The three local tables are substitutes, not paper Table 2 datasets. Bank/Table 2 has a required scale-up after EXIT.
The paper-results run is GPU recommended; use persistent storage and Modal for unattended long runs.

### Running on Google Colab?

Colab opens only this single file, so the course package (`relkit`) and the lab
dependencies (xgboost, lightgbm, catboost, …) are **not** present by default. The cell
below fixes that: on Colab it shallow-clones the course repo, installs
`requirements-labs.txt`, and switches into `labs/` so `relkit` imports and the data cache
resolve. **On a local venv or your own Jupyter it does nothing — just run it and continue.**

In [ ]:
# @colab-bootstrap — PROVIDED. Makes the lab self-sufficient on Google Colab; a no-op elsewhere.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.isdir("/content/relational"):
        !git clone --depth 1 https://github.com/Avistian/relational.git /content/relational
    %pip install -q -r /content/relational/requirements-labs.txt
    os.chdir("/content/relational/labs")
    print("Colab ready — working dir:", os.getcwd())
else:
    print("Not on Colab — using the local environment as-is.")

## Concept recap — a batch becomes part of the input
A token is a learned vector for one feature. Let **B** be rows, **T** tokens per row including CLS, and **d** coordinates per token. Feature attention operates on `[B,T,d]`: rows remain independent. Intersample attention flattens each row into length `T*d`, then regards the **B rows as one sequence**, `[1,B,T*d]`. Restore the original shape afterward. CLS is a learned summary slot, never the target label.

For each head, `A = softmax(QKᵀ / sqrt(head_dim))`, normalized over keys. Queries ask what to retrieve; keys determine match strength; values carry the information. A weighted sum changes a query even when only a companion changes. For scores `[1,0]` and values `[2,8]`, the weights are approximately `[0.731,0.269]`, and the output is `3.614`. With only the query present the sole weight is 1 and the value is 2.

The contrastive term asks each clean row to identify its own augmented view. Its positive is the matching row index, not a different row with the same class. With B equally likely candidates, mean loss is `log(B)`. Denoising separately reconstructs original features from corrupted views; our small loss exercise does not implement that whole pretraining experiment.

**Code/prose audit:** the released block computes `LN(x) + F(LN(x))`, uses GEGLU feed-forward gates and per-numeric `1→100→d` MLPs. These details differ from the PDF’s equations/prose. We match the released stage, with exact transplanted-weight forward and input-gradient checks; we do not silently call the different descriptions identical.

In [ ]:
# PROVIDED — reproducible environment; run locally from labs/ or labs/solutions/
import os, sys, json
from pathlib import Path
os.environ.setdefault('OMP_NUM_THREADS','1')
for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd()/'labs'):
    if (candidate/'relkit').is_dir():
        sys.path.insert(0,str(candidate)); LABS=candidate.resolve(); break
import numpy as np
import torch
from torch.nn import functional as F
torch.set_num_threads(1)
from relkit.saint_experiment import prepare, environment
print(json.dumps(environment(),indent=2))

## Task 1 — pack and restore entire rows (Algorithm 1)
**Goal:** convert feature-token batches into a single sequence of flattened rows and back.
**Why:** a transpose that attends down each column implements a different model. Preserve every feature coordinate and include CLS.
Write the two reshape operations. The checks use distinct coordinates so an accidental permutation cannot pass.

In [ ]:
# TODO — two focused shape operations
def pack_rows(tokens):
    b,t,d=tokens.shape
    return ____

def unpack_rows(rows,n_tokens):
    return ____


In [ ]:
# CHECK — values and row ownership, not just shape
x=torch.arange(60.).reshape(3,4,5)
packed=pack_rows(x)
assert packed.shape==(1,3,20)
assert torch.equal(packed[0,1],x[1].flatten())
assert torch.equal(unpack_rows(packed,4),x)
print('PASS: entire-row packing is invertible')

## Task 2 — normalize the attention over candidate rows
**Goal:** implement score scaling and softmax over keys.
**Why:** query normalization has the right tensor shape but fails to form a distribution over companions.
The last axis of Q/K is the head width, not the number of rows or feature tokens.

In [ ]:
# TODO — attention probabilities
def attention_weights(q,k):
    scores=____
    return ____


In [ ]:
# CHECK — an independent PyTorch kernel validates your calculation
q,k,v=[torch.randn(2,3,5,7,dtype=torch.float64) for _ in range(3)]
a=attention_weights(q,k)
assert torch.allclose(a.sum(-1),torch.ones_like(a.sum(-1)))
assert torch.allclose(a@v,F.scaled_dot_product_attention(q,k,v),atol=1e-12)
print('PASS: attention agrees with independent reference')

## Task 3 — the contrastive pairing (Eq.5, first term)
**Goal:** use each original row’s index as the correct augmented-view class.
**Why:** class labels are unnecessary for this task. A shuffled positive pairing trains the wrong invariance.
Use cross-entropy on the similarity matrix scaled by temperature. Mean loss is a batch-size-independent rescaling of the paper’s sum. These input vectors are not implicitly normalized; normalization is a separate design choice.

In [ ]:
# TODO — contrastive loss; no dataset labels enter this function
def info_nce(z,z_view,temperature=.7):
    logits=____
    return ____


In [ ]:
# CHECK — correct pairing beats a mismatched view; equal logits imply log(B)
z=torch.eye(4)
assert info_nce(z,z)<info_nce(z,z.roll(1,0))
assert abs(float(info_nce(torch.zeros(4,3),torch.zeros(4,3)))-np.log(4))<1e-6
print('PASS: identity pairing and uniform-loss baseline')

## PROVIDED — the full supervised implementation
Read the numeric embedding, both attention blocks, normalization wrapper, missing-value token, CLS head, and training loop below.
Your three functions remain in use: this source copy removes their canonical definitions. The model you train calls your implementations.
The exact reference-stage check is regenerable with `_check_l047.py --reference PATH`; see the reproduction reference for the pinned source command.

In [ ]:
# PROVIDED — inlined from `labs/relkit/saint.py` so you can read every line.
# This is the paper implementation, not `import relkit...` hiding it.
# Canonical file stays at labs/relkit/saint.py for Modal / _verify; this cell is a copy.

"""L047: supervised SAINT, written from scratch against the released code.

Paper: Somepalli et al., arXiv:2106.01342v1, §3 / Algorithm 1.
Reference: somepago/saint e288e84c77a54cfd2ffb55a53678fb7cbbb16630.
We deliberately match released PreNorm(Residual(F)): LN(x) + F(LN(x)),
NOT the paper's LN(F(x)) + x. Numeric embeddings follow released 1→100→d
MLPs. Attention dropout is absent in released forward; FF dropout is active.
The supervised path is complete; InfoNCE is an isolated §4 exercise, not a
claim that we reproduced the semi-supervised pretraining experiments.
"""
from __future__ import annotations

import copy
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from sklearn.metrics import roc_auc_score


class Attention(nn.Module):
    def __init__(self, width, heads=4, head_dim=16):
        super().__init__()
        self.heads, self.head_dim = heads, head_dim
        self.to_qkv = nn.Linear(width, 3 * heads * head_dim, bias=False)
        self.to_out = nn.Linear(heads * head_dim, width)

    def forward(self, x):
        b, t, _ = x.shape
        q, k, v = [v.reshape(b, t, self.heads, self.head_dim).transpose(1, 2)
                   for v in self.to_qkv(x).chunk(3, dim=-1)]
        a = attention_weights(q, k)
        return self.to_out((a @ v).transpose(1, 2).reshape(b, t, -1))


class GEGLU(nn.Module):
    def forward(self, x):
        value, gate = x.chunk(2, dim=-1)
        return value * F.gelu(gate)


class ReleasedResidual(nn.Module):
    """Mirror the executable nesting, not a conventional PreNorm assumption."""
    def __init__(self, width, fn):
        super().__init__()
        self.norm, self.fn = nn.LayerNorm(width), fn

    def forward(self, x):
        u = self.norm(x)
        return u + self.fn(u)


def feedforward(width, dropout):
    return nn.Sequential(nn.Linear(width, 8 * width), GEGLU(),
                         nn.Dropout(dropout), nn.Linear(4 * width, width))


class SaintStage(nn.Module):
    """Fig.1a: feature attention/FF, then entire-row attention/FF."""
    def __init__(self, tokens, d, heads=4, ff_dropout=0.1, variant='colrow'):
        super().__init__()
        if variant not in ('col', 'row', 'colrow'):
            raise ValueError(variant)
        self.variant = variant
        self.col = nn.ModuleList([
            ReleasedResidual(d, Attention(d, heads, 16)),
            ReleasedResidual(d, feedforward(d, ff_dropout)),
        ]) if variant != 'row' else nn.ModuleList()
        width = tokens * d
        self.row = nn.ModuleList([
            ReleasedResidual(width, Attention(width, heads, 64)),
            ReleasedResidual(width, feedforward(width, ff_dropout)),
        ]) if variant != 'col' else nn.ModuleList()

    def forward(self, x):
        for layer in self.col:
            x = layer(x)
        if self.row:
            t = x.shape[1]
            x = pack_rows(x)
            for layer in self.row:
                x = layer(x)
            x = unpack_rows(x, t)
        return x


class SAINT(nn.Module):
    """Released supervised path: CLS, cats, numeric MLPs, blocks, CLS→1000→2.
    Missing numeric inputs are NaN and get feature-specific learned mask tokens;
    categorical code 0 is the missing/unseen token reserved by our train-only encoder.
    """
    def __init__(self, n_num, cards, d=8, depth=1, heads=4,
                 ff_dropout=0.1, variant='colrow'):
        super().__init__()
        self.cls = nn.Parameter(torch.randn(1, 1, d))
        self.cats = nn.ModuleList([nn.Embedding(c, d) for c in cards])
        self.nums = nn.ModuleList([nn.Sequential(nn.Linear(1, 100), nn.ReLU(),
                                                nn.Linear(100, d)) for _ in range(n_num)])
        self.missing_num = nn.Parameter(torch.randn(n_num, d))
        t = 1 + n_num + len(cards)
        self.stages = nn.ModuleList([SaintStage(t, d, heads, ff_dropout, variant)
                                     for _ in range(depth)])
        self.head = nn.Sequential(nn.Linear(d, 1000), nn.ReLU(), nn.Linear(1000, 2))

    def tokenize(self, x_num, x_cat):
        b = len(x_num)
        parts = [self.cls.expand(b, -1, -1)]
        parts += [emb(x_cat[:, j]).unsqueeze(1) for j, emb in enumerate(self.cats)]
        for j, mlp in enumerate(self.nums):
            value = x_num[:, j:j+1]
            embedded = mlp(torch.nan_to_num(value))
            parts.append(torch.where(torch.isnan(value), self.missing_num[j], embedded).unsqueeze(1))
        return torch.cat(parts, dim=1)

    def encode(self, x_num, x_cat):
        x = self.tokenize(x_num, x_cat)
        for stage in self.stages:
            x = stage(x)
        return x

    def forward(self, x_num, x_cat):
        return self.head(self.encode(x_num, x_cat)[:, 0])


@torch.no_grad()
def predict_saint(model, xn, xc, batch_size=64, device='cpu'):
    """Within-split sequential batches. Their order/membership are model inputs.
    Never mix validation and test rows, and never pass labels into forward.
    """
    model.eval()
    out = []
    for start in range(0, len(xn), batch_size):
        num = torch.as_tensor(xn[start:start+batch_size], dtype=torch.float32, device=device)
        cat = torch.as_tensor(xc[start:start+batch_size], dtype=torch.long, device=device)
        out.append(model(num, cat).softmax(-1)[:, 1].cpu().numpy())
    return np.concatenate(out)


def train_saint(model, xn, xc, y, train, valid, *, seed=0, epochs=20,
                batch_size=64, lr=1e-3, device='cpu', checkpoint=None,
                select_metric='auc', validate_every=1):
    """Supervised AdamW, validation-only selection, checkpoint best weights.
    Call torch.manual_seed BEFORE constructing the model as well as here.
    No early stopping: a fixed budget makes the local ablation interpretable.
    """
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    num = torch.as_tensor(xn, dtype=torch.float32, device=device)
    cat = torch.as_tensor(xc, dtype=torch.long, device=device)
    labels = torch.as_tensor(y, dtype=torch.long, device=device)
    best, best_score, history = None, -float('inf'), []
    for epoch in range(epochs):
        model.train()
        order = rng.permutation(train)
        for start in range(0, len(order), batch_size):
            idx = order[start:start+batch_size]
            optimizer.zero_grad()
            loss = F.cross_entropy(model(num[idx], cat[idx]), labels[idx])
            loss.backward()
            optimizer.step()
        if epoch % validate_every == 0:
            p = predict_saint(model, xn[valid], xc[valid], batch_size, device)
            score = (float(np.mean((p >= .5) == y[valid])) if select_metric == 'accuracy'
                     else float(roc_auc_score(y[valid], p)))
            history.append({'epoch': epoch + 1, 'valid_score': score})
            if score > best_score:
                best_score, best = score, copy.deepcopy(model.state_dict())
                if checkpoint:
                    torch.save({'weights': best, 'epoch': epoch + 1, 'seed': seed,
                                'valid_score': score}, checkpoint)
    model.load_state_dict(best)
    return model, history


## Task 4 — intervene on a companion, holding the query fixed
**Goal:** distinguish permutation equivariance from independence of batch membership.
**Why:** reordering an unchanged batch should reorder outputs; changing its members can change them.
Both models are in evaluation mode with dropout disabled for this probe. The tensors are synthetic only to isolate the operator; the next experiment trains on real data.

In [ ]:
# PROVIDED + CHECK — causal wiring probe, not an accuracy claim
torch.manual_seed(47)
x=torch.randn(5,4,8)
row=SaintStage(4,8,ff_dropout=0.).eval()
col=SaintStage(4,8,ff_dropout=0.,variant='col').eval()
changed=x.clone(); changed[1]=torch.randn_like(x[1])*3
with torch.no_grad():
    delta=(row(x)[0]-row(changed)[0]).abs().max().item()
    assert delta>1e-6
    assert torch.allclose(col(x)[0],col(changed)[0])
    perm=torch.tensor([3,1,4,0,2])
    assert torch.allclose(row(x)[perm],row(x[perm]),atol=1e-5)
print('PASS: different companions move query; row permutations preserve correspondence.',delta)

## Real-data ablation — run your model on three datasets × three seeds
**Held fixed:** split seed 5; 65/15/20 stratified split; preprocessing fitted on training rows; d=8, one stage, four heads, FF dropout .1; 20 epochs; AdamW .001; batch 64; validation AUC selects the checkpoint. Model seeds are 0,1,2 and set before construction.
**Varied:** feature-only vs feature+row attention. This also changes parameter count: it is an ablation of adding the row block, not a parameter-matched proof about attention alone. The feature-only control is depth-matched, not the paper’s six-stage SAINT-s.
**External baseline:** CatBoost, 300 iterations/depth 6/lr .05, same splits, validation AUC. It is an untuned budgeted baseline; no claim about the best achievable tree score.
**Measured:** AUROC (ranking quality for binary classes), seed SD and conditional seed CIs, per-dataset ranks, Friedman/Nemenyi. Three datasets provide little power. No test labels enter forward, and test metrics never choose checkpoints.

Tier A substitutes: credit_g, diabetes, blood_transfusion. Credit_g is not the paper’s Credit card-fraud table. The paper’s Bank, Blastchar, Arrhythmia, Arcene, Forest, Shoppers, Income, HTRU2, KDD99, Philippine, QSAR Bio, Shrutime, Spambase, Credit, Volkert and MNIST are not this local benchmark.

In [ ]:
# PROVIDED — harness imports are allowed; your inlined model/train/predict functions are injected
from _verify_l047 import run
result=run(model_cls=SAINT,train_fn=train_saint,predict_fn=predict_saint)
print(json.dumps(result['results'],indent=2))
Path('l047_student_results.json').write_text(json.dumps(result,indent=2))

In [ ]:
# CHECK — completeness and context audit, with no enforced winning model
assert len(result['results']['per_dataset'])==3
for dataset,arms in result['results']['per_dataset'].items():
    assert all(len(a['scores'])==3 for a in arms.values())
for name,probe in result['context_probe'].items():
    if name.endswith('/col'):
        assert probe['max_abs_probability_change']<1e-5
assert all(set(p['train']).isdisjoint(p['test']) for p in result['protocols'].values())
print('PASS: three datasets, three seeds, split boundaries, independent-row control')

## EXIT TICKET
Explain the reshape, one companion intervention, and one limitation of the local comparison.
Report the actual winner/tie you measured; no CHECK requires SAINT to win. Distinguish failure to reject a rank-test null from proof of equivalence. Paste the output plus your explanation to the teacher; follow-up questions are welcome.

In [ ]:
# EXIT — evidence, not a prewritten conclusion
print('Mean ranks:',result['results']['mean_ranks'])
print('Friedman:',result['results']['friedman'])
print('Nemenyi CD:',result['results']['nemenyi_cd'])
print('Batch-context probe:',json.dumps(result['context_probe']['diabetes/0/colrow'],indent=2))
print('Paper reproduction status: NOT_RUN until the required next step executes.')

## REQUIRED NEXT STEP — attempt the paper’s supervised Bank result
The paper’s **Table 2 Bank AUROC is 0.9330**, five-trial mean. An absolute tolerance of ±0.01 would be a reproduction target only after agreeing the protocol. Our honest status remains INCOMPARABLE when unresolved split/code differences remain, even if the number happens to match.

The same from-scratch implementation and loop run below. `smoke` = 600 rows/1 epoch/1 seed (pipeline check); `closer` = full Bank/20 epochs/3 seeds/d=16; `paper` = full Bank/100 epochs/5 seeds/d=32/8 heads. The last preset adopts paper budgets, not a certificate of exact reproduction. No full pretraining benchmark is included.

Read `GAPS` and the three-bucket ledger before interpreting results. Set `RUN_PAPER_REPRO=True` for a deliberate run. In Colab, attach a GPU and mount Drive; set `OUTPUT_DIR` to a persistent Drive directory. Completed seeds can be resumed; interrupted seeds restart. The best checkpoint is saved for inference, not full optimizer resume. For multi-hour unattended runs use:

```bash
modal run --detach modal/l047_paper_repro.py --preset closer
modal volume get relational-artifacts l047/closer ./l047-bank-results
```

Runtime of the full presets is unmeasured; do not assume it fits a free Colab session. Start with smoke and budget from the measured time. Test predictions, split IDs, preprocessing, versions, seeds, and best weights are saved. Until the larger run finishes, the Table 2 claim stays cited, not reproduced.

In [ ]:
# PROVIDED — inlined from `labs/_paper_repro_l047.py` so you can read every line.
# This is the paper implementation, not `import relkit...` hiding it.
# Canonical file stays at labs/_paper_repro_l047.py for Modal / _verify; this cell is a copy.

"""SAINT supervised Bank/Table 2 attempt. Presets do not certify a protocol match.
Run: python labs/_paper_repro_l047.py --preset smoke --output-dir /tmp/l047_smoke
"""
from __future__ import annotations
import argparse
import hashlib
import json
import os
from pathlib import Path
import time
import numpy as np
import torch
from sklearn.metrics import roc_auc_score
from relkit.saint_experiment import prepare, environment
from relkit.paper_repro import LabFinding, PaperTarget, ScaleUpRun, classify_number, format_ledger

TARGET = PaperTarget(paper='SAINT (Somepalli et al., 2021)',arxiv='2106.01342',
    table='Table 2, Bank, supervised SAINT',dataset='Bank (OpenML 1461)',metric='AUROC',
    paper_value=.9330,abs_tol=.01,
    notes='Five-trial mean; not the 14-dataset mean 0.9313. Table 6 reports SE 0.0009, not SD.')
PRESETS = {
    'smoke':dict(cap=600,epochs=1,d=4,heads=2,seeds=[0],batch_size=32,ff_dropout=.1,lr=.001),
    'closer':dict(cap=None,epochs=20,d=16,heads=4,seeds=[0,1,2],batch_size=256,ff_dropout=.8,lr=.0001),
    'paper':dict(cap=None,epochs=100,d=32,heads=8,seeds=[0,1,2,3,4],batch_size=256,ff_dropout=.8,lr=.0001),
}
GAPS = [
    'Original five split-index files unavailable: released 65/15/20 assignment, split seed 5.',
    'Paper says 65/15/25 (105%); released data_openml.py uses 65/15/20.',
    'Released normalization LN(x)+F(LN(x)), GEGLU, numeric 1→100→d; prose describes different placement/embedding.',
    'Training-only category vocabulary with reserved unknown; released encoder fits vocabulary to full table.',
    'Paper uses 8 heads; current train.py clamps combined variant to 4. paper preset chooses paper count.',
    'Attention dropout declared but unused in released forward; feed-forward dropout is applied.',
    'Bank only; no reproduction of 14-task mean, semi-supervised Table 3, or superiority over tuned trees.',
]


def main(argv=None):
    parser=argparse.ArgumentParser()
    parser.add_argument('--preset',choices=PRESETS,default='closer')
    parser.add_argument('--output-dir',default=os.environ.get('L047_OUTPUT','l047_artifacts'))
    args=parser.parse_args(argv); cfg=PRESETS[args.preset]
    dest=Path(args.output_dir); dest.mkdir(parents=True,exist_ok=True)
    torch.set_num_threads(1)
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
    torch.use_deterministic_algorithms(True)
    dev='cuda' if torch.cuda.is_available() else 'cpu'
    fr=prepare('bank_marketing',cap=cfg['cap'],split='released')
    (dest/'protocol.json').write_text(json.dumps(fr['meta'],indent=2))
    from relkit import saint_experiment
    source_dir = Path(saint_experiment.__file__).resolve().parents[1]
    source_hashes = {name: hashlib.sha256((source_dir/name).read_bytes()).hexdigest()
                     for name in ('relkit/saint.py','relkit/saint_experiment.py','_paper_repro_l047.py')}
    values=[]; start=time.time()
    for seed in cfg['seeds']:
        artifact=dest/f'{args.preset}_seed{seed}.json'
        identity=hashlib.sha256(json.dumps({'cfg':cfg,'data':fr['meta']['data_sha256'],
            'split':fr['meta']['split_hashes'],'seed':seed,'source_hashes':source_hashes,'environment':environment()},sort_keys=True).encode()).hexdigest()
        if artifact.exists():
            record=json.loads(artifact.read_text())
            if record.get('identity') != identity:
                raise RuntimeError(f'{artifact} belongs to a different run; use a fresh output directory')
        else:
            torch.manual_seed(seed)
            model=SAINT(fr['xn'].shape[1],fr['cards'],d=cfg['d'],heads=cfg['heads'],
                        ff_dropout=cfg['ff_dropout'],variant='colrow')
            model,hist=train_saint(model,fr['xn'],fr['xc'],fr['y'],fr['train'],fr['valid'],seed=seed,
                epochs=cfg['epochs'],batch_size=cfg['batch_size'],lr=cfg['lr'],device=dev,
                checkpoint=str(dest/f'{args.preset}_seed{seed}_best.pt'),
                select_metric='accuracy',validate_every=5)
            te=fr['test']; pred=predict_saint(model,fr['xn'][te],fr['xc'][te],cfg['batch_size'],dev)
            np.savez(dest/f'{args.preset}_seed{seed}_predictions.npz',row_ids=te,y=fr['y'][te],p=pred)
            record={'identity':identity,'seed':seed,'auc':float(roc_auc_score(fr['y'][te],pred)),
                    'history':hist,'environment':environment()}
            artifact.write_text(json.dumps(record,indent=2))
        values.append(record['auc']); print(f'Bank seed {seed}: AUROC {values[-1]:.6f}',flush=True)
    gaps=GAPS+[f'Preset {args.preset}: {cfg}. Selection follows released binary code: validation accuracy every fifth epoch.']
    run=ScaleUpRun(method='from-scratch SAINT',dataset=TARGET.dataset,metric='AUROC',
        value=float(np.mean(values)),std=float(np.std(values,ddof=1)) if len(values)>1 else None,
        n_seeds=len(values),hardware=str(torch.cuda.get_device_name(0)) if dev=='cuda' else 'CPU',
        wall_s=time.time()-start,protocol_match=False,protocol_deviations=gaps)
    ledger=format_ledger(title='L047 supervised SAINT / Bank',
        lab=[LabFinding('Row-context mechanism and local ablation','See _check_l047_results.json and _verify_l047_results.json',
                        'Three substitute datasets; no pretraining')],
        paper=[(TARGET,run,classify_number(TARGET,run))],
        extra_lines=[f'Descriptive delta from Bank 0.9330: {run.value-.9330:+.6f}; ±0.01 tolerance applies only after protocol agreement.',
                     'Completed seeds resume; interrupted seeds restart. Best weights support inference, not optimizer resume.'])
    out={'preset':args.preset,'source_hashes':source_hashes,'config':cfg,'environment':environment(),'scores':values,'ledger':ledger}
    (dest/f'{args.preset}_results.json').write_text(json.dumps(out,indent=2))
    print(ledger); return out


In [ ]:
# PROVIDED — explicit compute gate
RUN_PAPER_REPRO=False
PRESET='closer'
OUTPUT_DIR='l047_artifacts'  # Colab: change to a mounted persistent Drive path
if RUN_PAPER_REPRO:
    paper_result=main(['--preset',PRESET,'--output-dir',OUTPUT_DIR])
else:
    print('NOT_RUN: paper-scale Bank. Run smoke first; then use Colab GPU or the Modal command above.')